# Tiền xử lý cho PhoBERT — tách từ

**Mục đích:** tạo cột `comment_seg` (văn bản đã tách từ) từ `comment_norm`, lưu vào `artifacts/phobert/` để notebook fine-tune dùng.

**Vì sao cần tách từ:** PhoBERT được pretrain trên văn bản đã tách từ bằng VnCoreNLP. Tài liệu chính thức ghi rõ: *"INPUT TEXT MUST BE ALREADY WORD-SEGMENTED!"*.

**Chọn công cụ:** `underthesea` (pure Python, không cần Java). Nếu muốn đúng bài báo hơn, có thể dùng `py_vncorenlp` (cần JDK 17; máy đang có Java 25 nên có thể phải hạ phiên bản).

**Cài đặt:**

```powershell
python -m pip install underthesea
```

**Đầu ra:**
- `artifacts/phobert/train.parquet`
- `artifacts/phobert/dev.parquet`
- `artifacts/phobert/test.parquet`

Mỗi file gồm: `comment_norm`, `comment_seg`, và `spam_label`/`id` nếu có.

**Không đụng** `X_*.npz` hay bất kỳ artifact cũ nào.

In [1]:
# ============================================================
# 1. IMPORT & KIỂM TRA
# ============================================================
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 160)


def find_project_root(start=None, marker="Data/ds102_train.csv"):
    p = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}'")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
CLEAN_DIR = ARTIFACT_DIR / "clean"
PHOBERT_DIR = ARTIFACT_DIR / "phobert"
PHOBERT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from underthesea import word_tokenize
    print("underthesea: OK")
except ImportError:
    raise ImportError(
        "Chưa cài underthesea. Chạy:\n"
        "    python -m pip install underthesea"
    ) from None

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PHOBERT_DIR :", PHOBERT_DIR)

underthesea: OK
PROJECT_ROOT: D:\DS102_NguyenKhacHuy_24520679\Final project
PHOBERT_DIR : D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\phobert


In [2]:
# ============================================================
# 2. NẠP DỮ LIỆU ĐÃ LÀM SẠCH
# ============================================================
frames = {}
for split in ["train", "dev", "test"]:
    path = CLEAN_DIR / f"clean_{split}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Thiếu {path}. Hãy chạy src/preprocessing.ipynb trước.")
    frames[split] = pd.read_parquet(path)
    print(f"{split:5s} rows={len(frames[split]):5d}  cols={len(frames[split].columns)}")

assert "comment_norm" in frames["train"].columns
print("\nVí dụ comment_norm:")
print(frames["train"]["comment_norm"].head(3).to_string())

train rows=14306  cols=20
dev   rows= 1590  cols=20
test  rows= 3974  cols=20

Ví dụ comment_norm:
0                                                                                                      sản phẩm đúng quảng cáo. giao hàng hơi chậm nhg vẫn cho 5 sao
1    Mình nhận hàng ngày hôm qua, tiki giao hàng đúng hẹn dù ảnh hưởng bởi giãn cách theo chỉ thị 16, nhân viên giao hàng mang sản phẩm của mình vào tận nhà, tu ...
2                                                                        Máy giặt quá tốt, chỉ có 5 tr được cái máy giặt lồng ngang, ít nhăn hơn máy giặt lồng đứng.


## 3. Tách từ

`word_tokenize(text, format="text")` nối các từ ghép bằng dấu `_`, ví dụ:

```
"giao hàng nhanh" → "giao_hàng nhanh"
```

Cell dưới chạy cho cả 3 split (~20k câu, vài phút) và lưu parquet.

In [3]:
# ============================================================
# 3. TÁCH TỪ & LƯU
# ============================================================
def segment(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    return word_tokenize(text, format="text")


t_all = time.time()
for split, df in frames.items():
    t0 = time.time()
    df["comment_seg"] = df["comment_norm"].map(segment)

    out = df[["comment_norm", "comment_seg"]].copy()
    if "spam_label" in df.columns:
        out["spam_label"] = df["spam_label"].to_numpy()
    if "id" in df.columns:
        out["id"] = df["id"].to_numpy()

    path = PHOBERT_DIR / f"{split}.parquet"
    out.to_parquet(path, index=False)
    print(f"{split:5s} xong trong {time.time() - t0:.0f}s → {path}")

print(f"\nTổng thời gian: {time.time() - t_all:.0f}s")

train xong trong 23s → D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\phobert\train.parquet
dev   xong trong 3s → D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\phobert\dev.parquet
test  xong trong 6s → D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\phobert\test.parquet

Tổng thời gian: 33s


## 4. Kiểm tra chất lượng tách từ

Xem vài ví dụ và độ dài token (PhoBERT giới hạn 256 token; ta thường dùng 128).

In [4]:
# ============================================================
# 4. KIỂM TRA
# ============================================================
print("Ví dụ trước / sau tách từ:")
for i in range(3):
    print("-" * 90)
    print("TRƯỚC:", frames["train"]["comment_norm"].iloc[i][:160])
    print("SAU  :", frames["train"]["comment_seg"].iloc[i][:160])

print("\nĐộ dài (số token tách bằng khoảng trắng):")
for split, df in frames.items():
    lengths = df["comment_seg"].str.split().str.len()
    print(f"  {split:5s} mean={lengths.mean():.1f}  median={lengths.median():.0f}  "
          f"max={lengths.max()}  >128={(lengths > 128).mean():.1%}  >256={(lengths > 256).mean():.1%}")

print("\nKiểm tra giữ nguyên dấu và chữ hoa:")
sample = frames["train"]["comment_norm"].iloc[0]
print("  norm  :", sample[:100])
print("  seg   :", frames["train"]["comment_seg"].iloc[0][:100])

Ví dụ trước / sau tách từ:
------------------------------------------------------------------------------------------
TRƯỚC: sản phẩm đúng quảng cáo. giao hàng hơi chậm nhg vẫn cho 5 sao
SAU  : sản_phẩm đúng quảng_cáo . giao hàng hơi chậm nhg vẫn cho 5 sao
------------------------------------------------------------------------------------------
TRƯỚC: Mình nhận hàng ngày hôm qua, tiki giao hàng đúng hẹn dù ảnh hưởng bởi giãn cách theo chỉ thị 16, nhân viên giao hàng mang sản phẩm của mình vào tận nhà, tu lạnh
SAU  : Mình nhận hàng ngày hôm_qua , tiki giao hàng đúng hẹn dù ảnh_hưởng bởi giãn cách theo chỉ_thị 16 , nhân_viên giao hàng mang sản_phẩm của mình vào tận nhà , tu_l
------------------------------------------------------------------------------------------
TRƯỚC: Máy giặt quá tốt, chỉ có 5 tr được cái máy giặt lồng ngang, ít nhăn hơn máy giặt lồng đứng.
SAU  : Máy giặt quá tốt , chỉ có 5 tr được cái máy_giặt lồng ngang , ít nhăn hơn máy giặt lồng đứng .

Độ dài (số token tách b

## 5. Ghi chú

- Không lower-case, không bỏ stopword, không bỏ emoji/link — PhoBERT cần ngữ cảnh đầy đủ.
- Nếu sau này dùng `py_vncorenlp`, chỉ cần đổi hàm `segment()` và chạy lại cell 3; định dạng đầu ra giữ nguyên.
- Notebook fine-tune (`phobert_finetune.ipynb`) sẽ đọc 3 file parquet này.